# 09 — Phân tích định lượng đặc trưng (EDA), Stress-Test & Đánh giá MRR@5 (Milestone 3)

Notebook này trình bày toàn bộ phân tích dữ liệu định lượng phục vụ **Milestone 3 (Reference Matching Pipeline)** của đồ án **arXiv Harvester** (25.000 bài báo / 41.498 phiên bản):

1. **Kiểm kê tập gán nhãn (Manual + Automatic)** theo quy tắc từng thành viên (`k = 5`).
2. **Phân tích chất lượng metadata BibTeX & độ trễ năm (`bib_year - ref_year`)** trên `29.309` mục BibTeX (`1.115.447` cặp ứng viên thật) để biện giải động lực thiết kế 25 đặc trưng cổ điển.
3. **Sức phân tách đơn biến (ROC-AUC & Positive vs Negative)** của cả 5 nhóm đặc trưng — bao gồm **4 đặc trưng ngữ cảnh cây Hierarchy M2 (`elements` + `hierarchy`)**.
4. **So sánh 4 mô hình cổ điển (không dùng LLM/pretrained)** và **Nghiên cứu loại bỏ đặc trưng (Ablation A1 → A2 → A3)** trên cả tập **Test chuẩn** lẫn **Stress-Test `.bbl` nhiễu/khuyết (S2, S3)** và **5-Fold Cross-Validation trên 25 bài Manual**.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

WORKSPACE = Path('../build/m3_workspace')
m3_summary = json.loads((WORKSPACE / 'm3_results_summary.json').read_text(encoding='utf-8'))
m3_eda = json.loads((WORKSPACE / 'm3_feature_eda.json').read_text(encoding='utf-8'))
m3_ablation = json.loads((WORKSPACE / 'm3_ablation.json').read_text(encoding='utf-8'))

lbl = m3_summary['labeling']
print(f"Manual pool : {lbl['manual_papers']} bài ({lbl['manual_entries']:,} mục, {lbl['manual_positive_entries']:,} dương)")
print(f"Auto pool   : {lbl['auto_papers']:,} bài ({lbl['auto_fraction_of_remaining']*100:.1f}% > 10%, {lbl['auto_entries']:,} mục)")
print(f"Split entries: Train={lbl['split_entries']['train']:,} | Valid={lbl['split_entries']['valid']:,} | Test={lbl['split_entries']['test']:,}")
print(f"EDA sample  : {m3_eda['n_papers_analyzed']} bài | {m3_eda['n_candidate_pairs_analyzed']:,} cặp ({m3_eda['n_positive_pairs_analyzed']:,} cặp dương)")

Manual pool : 25 bài (1,150 mục, 1,003 dương)
Auto pool   : 2,997 bài (12.0% > 10%, 161,319 mục)
Split entries: Train=161,515 | Valid=414 | Test=540
EDA sample  : 535 bài | 1,115,447 cặp (10,831 cặp dương)


## 1. Phân tích chất lượng BibTeX & Bất đối xứng độ trễ năm (`bib_year - ref_year`)

Số liệu dưới đây lý giải trực tiếp tại sao không thể chỉ dùng khớp chuỗi tuyệt đối hay regex arXiv ID:
- Hầu hết mục BibTeX thiếu `doi`/`url` và chỉ ~10% mang sẵn arXiv ID tường minh.
- Gần 24% cặp đúng bị lệch số lượng tác giả do rút gọn `et al.`.
- Độ trễ năm mang tính **bất đối xứng mạnh** (`+1 năm` cao gấp ~95 lần `-1 năm`) do quy trình nộp preprint arXiv trước khi đăng hội nghị/tạp chí.

In [2]:
meda = m3_eda['metadata_and_year_lag_eda']
meta_df = pd.DataFrame([
    ('Thiếu DOI trong BibTeX', f"{meda['missing_doi_pct']:.2f}%"),
    ('Thiếu URL trong BibTeX', f"{meda['missing_url_pct']:.2f}%"),
    ('Có sẵn arXiv ID tường minh', f"{meda['has_explicit_arxiv_id_pct']:.2f}%"),
    ('Giải được ngữ cảnh \\cite{} từ cây M2', f"{meda['has_m2_hierarchy_ctx_pct']:.2f}%"),
    ('Xuất hiện trong Intro / Related Work (M2)', f"{meda['cited_in_intro_or_related_pct']:.2f}%"),
    ('Cặp dương khớp tiêu đề chuẩn hóa tuyệt đối', f"{meda['positive_exact_title_pct']:.2f}%"),
    ('Cặp dương bị lệch số lượng tác giả (et al.)', f"{meda['author_count_mismatch_pct']:.2f}% (TB lệch {meda['mean_author_count_diff']:.2f})"),
], columns=['Chỉ tiêu chất lượng metadata', 'Giá trị đo thực tế'])
print(meta_df.to_string(index=False))

lag_df = pd.DataFrame([
    {'Độ trễ năm (bib - ref)': int(k), 'Số cặp dương': v['count'], 'Tỷ lệ (%)': v['pct']}
    for k, v in meda['year_lag_distribution'].items()
])
print('\nPhân bố độ trễ năm trên các cặp dương:')
print(lag_df.to_string(index=False))

               Chỉ tiêu chất lượng metadata    Giá trị đo thực tế
                     Thiếu DOI trong BibTeX                70.78%
                     Thiếu URL trong BibTeX                88.99%
                 Có sẵn arXiv ID tường minh                10.11%
       Giải được ngữ cảnh \cite{} từ cây M2                98.05%
  Xuất hiện trong Intro / Related Work (M2)                59.99%
 Cặp dương khớp tiêu đề chuẩn hóa tuyệt đối                94.59%
Cặp dương bị lệch số lượng tác giả (et al.) 23.79% (TB lệch 4.89)

Phân bố độ trễ năm trên các cặp dương:
 Độ trễ năm (bib - ref)  Số cặp dương  Tỷ lệ (%)
                     -2             7       0.07
                     -1            32       0.30
                      0          6760      64.28
                      1          3030      28.81
                      2           418       3.97
                      3           137       1.30
                      4            43       0.41


## 2. Sức phân tách đơn biến (ROC-AUC) của 25 đặc trưng cổ điển

Mỗi đặc trưng được đánh giá trên **1.115.447 cặp ứng viên thật** (10.831 dương vs 1.104.616 âm) để kiểm chứng khách quan khả năng tách biệt giữa cặp đúng và cặp nhiễu trong cùng bài báo, đặc biệt xác nhận đóng góp thực sự của nhóm **Ngữ cảnh cây Hierarchy M2**.

In [3]:
auc_df = pd.DataFrame(m3_eda['feature_separation_auc'])[
    ['feature', 'group', 'roc_auc', 'pos_mean', 'neg_mean', 'pos_nonzero_pct', 'neg_nonzero_pct']
]
print(auc_df.to_string(index=False))

                        feature                group  roc_auc  pos_mean  neg_mean  pos_nonzero_pct  neg_nonzero_pct
            title_token_jaccard        base_metadata   0.9957    0.9819    0.0330            99.34            27.70
               title_token_dice        base_metadata   0.9957    0.9861    0.0566            99.34            27.70
             title_tfidf_cosine        base_metadata   0.9953    0.9871    0.0300            99.36            43.47
   title_containment_bib_in_ref        base_metadata   0.9952    0.9843    0.0595            99.34            27.70
          title_rapidfuzz_ratio        base_metadata   0.9942    0.9899    0.3227            99.88            83.58
          author_surname_recall        base_metadata   0.9913    0.9764    0.0329            99.26             9.65
         author_surname_jaccard        base_metadata   0.9910    0.8531    0.0155            99.26             9.65
           title_tfidf_rank_inv        pool_relative   0.9881    0.9937 

## 3. So sánh 4 mô hình cổ điển, Stress-Test `.bbl` & 5-Fold Manual CV (25 bài)

- Trên tập **Test chuẩn (`S0`)**, cả 4 mô hình đều đạt đúng trần lý thuyết **`MRR@5 = 0,690741`** (`373 / 540` vì `167` mục là hàng âm toàn phần không có bản arXiv; `100%` Recall@1 trên `373` mục dương).
- Khi kiểm định áp lực với `.bbl` rút gọn tiêu đề (`S2`) hoặc khuyết hoàn toàn tiêu đề (`S3`), **học máy cổ điển (`LightGBM Ranker`, `Random Forest`, `Logistic Regression`) vượt xa `Heuristic Baseline`** và **Ablation `A1 → A2 → A3` chứng minh rõ giá trị phục hồi của ngữ cảnh cây Hierarchy M2**.

In [4]:
models_order = ['heuristic_baseline', 'logistic_regression', 'random_forest', 'lightgbm_ranker']
rows = []
for m in models_order:
    rows.append({
        'Mô hình': m,
        'Test S0 MRR@5': m3_eda['stress_test']['S0_clean']['models'][m]['test']['mrr_at_5'],
        'Test S2 MRR@5': m3_eda['stress_test']['S2_noisy_bbl_title']['models'][m]['test']['mrr_at_5'],
        'Test S3 MRR@5': m3_eda['stress_test']['S3_missing_title_rescue']['models'][m]['test']['mrr_at_5'],
        'Test S3 Pos R@1': m3_eda['stress_test']['S3_missing_title_rescue']['models'][m]['test']['recall_at_1'],
        'CV5 S0 MRR@5': m3_eda['manual_5fold_cv']['S0_clean']['models'][m]['mrr_at_5_mean'],
        'CV5 S3 MRR@5': m3_eda['manual_5fold_cv']['S3_missing_title_rescue']['models'][m]['mrr_at_5_mean'],
        'CV5 S3 Pos R@1': m3_eda['manual_5fold_cv']['S3_missing_title_rescue']['models'][m]['recall_at_1_mean'],
    })
print('=== So sánh 4 mô hình cổ điển ===')
print(pd.DataFrame(rows).to_string(index=False))

abl_rows = []
for a in ('A1_base_metadata', 'A2_plus_pool_relative', 'A3_full_with_m2_hierarchy'):
    cv3 = m3_eda['manual_5fold_cv']['S3_missing_title_rescue']['ablation'][a]
    te3 = m3_eda['stress_test']['S3_missing_title_rescue']['ablation'][a]['test']
    abl_rows.append({
        'Cấu hình Ablation': a,
        'Test S3 MRR@5': te3['mrr_at_5'],
        'Test S3 Pos R@1': te3['recall_at_1'],
        'CV5 S3 MRR@5': cv3['mrr_at_5_mean'],
        'CV5 S3 Std': cv3['mrr_at_5_std'],
        'CV5 S3 Pos-MRR@5': cv3['mrr_pos_mean'],
        'CV5 S3 Pos R@1': cv3['recall_at_1_mean'],
        'CV5 S3 Pos R@5': cv3['recall_at_5_mean'],
    })
print('\n=== Ablation Study (S3 khuyết tiêu đề — LightGBM Ranker) ===')
print(pd.DataFrame(abl_rows).to_string(index=False))

=== So sánh 4 mô hình cổ điển ===
            Mô hình  Test S0 MRR@5  Test S2 MRR@5  Test S3 MRR@5  Test S3 Pos R@1  CV5 S0 MRR@5  CV5 S3 MRR@5  CV5 S3 Pos R@1
 heuristic_baseline       0.690741       0.673765       0.608611         0.809651        0.8717        0.7734          0.8144
logistic_regression       0.690741       0.684877       0.678704         0.967828        0.8721        0.8463          0.9462
      random_forest       0.690741       0.689815       0.675617         0.959786        0.8721        0.8524          0.9595
    lightgbm_ranker       0.690741       0.688889       0.680247         0.973190        0.8721        0.8530          0.9605

=== Ablation Study (S3 khuyết tiêu đề — LightGBM Ranker) ===
        Cấu hình Ablation  Test S3 MRR@5  Test S3 Pos R@1  CV5 S3 MRR@5  CV5 S3 Std  CV5 S3 Pos-MRR@5  CV5 S3 Pos R@1  CV5 S3 Pos R@5
         A1_base_metadata       0.679012         0.970509        0.8485      0.0188            0.9730          0.9551          0.9950
    A2

## 4. Kết luận

1. Toàn bộ pipeline Milestone 3 tuân thủ **100% học máy cổ điển** (TF-IDF + RapidFuzz + tác giả/năm + Pool-relative + ngữ cảnh cây Hierarchy M2; không dùng LLM hay pretrained model).
2. Bộ dữ liệu đầu ra (`predictions`, `162.469` dòng trên `3.022` bài) đạt chuẩn phẳng Parquet cho Hugging Face streaming cùng 5 bảng từ M1–M2.
3. Để chạy lại toàn bộ phân tích EDA & Stress-Test trên dữ liệu cục bộ:
   ```bash
   python -m matching.analysis --data-roots <hf_dataset_dir> --manual-labels <manual_labels.jsonl> --output m3_feature_eda.json
   ```